In [12]:
import os
import math
import numpy as np
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import t
from matplotlib.lines import Line2D
from lifelines import KaplanMeierFitter

USER_PATH = Path('/data/gusev/USERS/jpconnor/')
DATA_PATH = USER_PATH / 'data/CAIA/COMPASS/'
FIG_PATH = USER_PATH / 'figures/CAIA/COMPASS/ADT_METASTATIC_FILTERING/'

In [13]:
adt_df = pl.read_csv(DATA_PATH / 'survival_analysis/prediction_inputs_adt/aggregated_landmark0.csv')
localized_adt_df = pl.read_csv(DATA_PATH / 'survival_analysis/prediction_inputs_adt_localized/aggregated_landmark0.csv')
metastatic_adt_df = pl.read_csv(DATA_PATH / 'survival_analysis/prediction_inputs_adt_metastatic/aggregated_landmark0.csv')

adt_df = adt_df.with_columns(
    pl.when(pl.col("DFCI_MRN").is_in(metastatic_adt_df["DFCI_MRN"].to_list()))
    .then(pl.lit("Met"))
    .otherwise(pl.lit("Local"))
    .alias("METASTATIC_ADT")
)

In [14]:
for label, df in [('full', adt_df), ('localized', localized_adt_df), ('metastatic', metastatic_adt_df)]:
    print(f'label = {label}.') 
    print(f'# Platinum = {df['PLATINUM'].sum()}')
    print(f'# NEPC = {df['NEPC'].sum()}')
    print(f'# AVPC = {df['AVPC'].sum()}\n')

label = full.
# Platinum = 181
# NEPC = 147
# AVPC = 756

label = localized.
# Platinum = 6
# NEPC = 30
# AVPC = 175

label = metastatic.
# Platinum = 175
# NEPC = 117
# AVPC = 581



In [15]:
os.makedirs(FIG_PATH / 'KM_curves/', exist_ok=True)
for event in ['DEATH', 'PLATINUM', 'NEPC', 'AVPC']:
    fig, ax = plt.subplots(figsize=(8,6))
    
    for group, group_df in adt_df.group_by('METASTATIC_ADT'):
        kmf = KaplanMeierFitter()
        km_df = group_df.filter(pl.col(f't_{event.lower()}') > 0)
        kmf.fit(durations=km_df[f't_{event.lower()}'], event_observed=km_df[event], label=group)
        kmf.plot_survival_function(ax=ax, ci_show=True)
        
    ax.set(xlabel='Time (days)', ylabel=f'{event}-Free Probability', ylim=(0,1), title=f'{event} KM Curves by ADT Indication')
    ax.legend(title='ADT Indication')
    fig.tight_layout()
    fig.savefig(FIG_PATH / f'KM_curves/km_curves_by_ADT_indication_for_{event}.png', dpi=300, bbox_inches='tight')
    plt.close()

In [16]:
UNI_LAB_SCHEMA = {'cohort' : pl.String,
                  'landmark_days' : pl.Int16,
                  'endpoint' : pl.String,
                  'lab_name' : pl.String,
                  'feature_stat' : pl.String,
                  'coef_feature' : pl.Float64,
                  'hazard_ratio_per_sd' : pl.Float64,
                  'ci_lower' : pl.Float64,
                  'ci_upper' : pl.Float64,
                  'p_value' : pl.Float64,
                  'q_value' : pl.Float64,}

cox_lab_nominal_hits = (pl.scan_csv(DATA_PATH / 'survival_analysis/local_runs_adt*/cox/nominally_significant_univariate_results.csv',
                                   glob=True, schema_overrides=UNI_LAB_SCHEMA)
                        .filter(pl.col('endpoint').is_in(['platinum', 'avpc', 'nepc']))
                        .select(UNI_LAB_SCHEMA.keys())
                        .collect(engine='streaming'))

nominal_hits_counts = (cox_lab_nominal_hits
                       .group_by(['cohort', 'endpoint', 'landmark_days'])
                       .len(name='num_nominal_hits')
                       .sort('num_nominal_hits', descending=True))
corrected_hits_counts = (cox_lab_nominal_hits.filter(pl.col('q_value').cast(pl.Float64) < 0.05)
                         .group_by(['cohort', 'endpoint', 'landmark_days'])
                         .len(name='num_corrected_hits')
                         .sort('num_corrected_hits', descending=True))

full_hits_counts = (nominal_hits_counts
                    .join(corrected_hits_counts, on=['cohort', 'endpoint', 'landmark_days'], 
                          how='left', coalesce=True)
                    .with_columns(pl.col('num_corrected_hits').fill_null(0))
                    .sort('num_corrected_hits', descending=True))

In [17]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=1000, tbl_width_chars=-1,):
    print(full_hits_counts.filter(pl.col('landmark_days') == 0))
    print(full_hits_counts.filter(pl.col('landmark_days') == 90))
    print(full_hits_counts.filter(pl.col('landmark_days') == 180))

shape: (8, 5)
┌────────────────┬──────────┬───────────────┬──────────────────┬────────────────────┐
│ cohort         ┆ endpoint ┆ landmark_days ┆ num_nominal_hits ┆ num_corrected_hits │
│ ---            ┆ ---      ┆ ---           ┆ ---              ┆ ---                │
│ str            ┆ str      ┆ i16           ┆ u32              ┆ u32                │
╞════════════════╪══════════╪═══════════════╪══════════════════╪════════════════════╡
│ adt            ┆ platinum ┆ 0             ┆ 45               ┆ 31                 │
│ adt_metastatic ┆ platinum ┆ 0             ┆ 52               ┆ 29                 │
│ adt            ┆ avpc     ┆ 0             ┆ 35               ┆ 13                 │
│ adt            ┆ nepc     ┆ 0             ┆ 23               ┆ 5                  │
│ adt_localized  ┆ nepc     ┆ 0             ┆ 7                ┆ 3                  │
│ adt_metastatic ┆ nepc     ┆ 0             ┆ 25               ┆ 2                  │
│ adt_metastatic ┆ avpc     ┆ 0         

In [18]:
landmarks = [0, 90, 180]
feature_stats = ["delta", "mean", "min", "max", "last"]

colors = {
    "Not significant": "lightgray",
    "Nominal": "#F4A261",
    "Corrected": "#D62828",
}

def plot_HRs(nominal_hits_df, lab_name):
    lab_hits = (nominal_hits_df
                .filter(pl.col('lab_name') == lab_name)
                .with_columns(pl.when(pl.col('q_value') < 0.05)
                              .then(pl.lit('Corrected'))
                              .when(pl.col('p_value') < 0.05)
                              .then(pl.lit('Nominal'))
                              .otherwise(pl.lit('Not sig.'))
                              .alias('significance')))
    endpoints = lab_hits.get_column('endpoint').drop_nulls().unique().sort().to_list()
    
    for endpoint_val in endpoints:
        plot_df = lab_hits.filter(pl.col('endpoint') == endpoint_val)
        cohorts = lab_hits.get_column('cohort').drop_nulls().unique().sort().to_list()
        
        row_grid = (pl.DataFrame({'cohort' : cohorts, 'cohort_order' : range(len(cohorts))})
                    .join(pl.DataFrame({'feature_stat' : feature_stats, 'stat_order' : range(len(feature_stats))}), how='cross')
                    .sort(['cohort_order', 'stat_order'])
                    .with_columns(pl.concat_str(['cohort', 'feature_stat'], separator=' - ').alias('row_label'))
                    .with_row_index('y'))
        full_grid = (pl.DataFrame({'landmark_days' : landmarks})
                     .join(row_grid, how='cross')
                     .join(plot_df, on=['landmark_days', 'cohort', 'feature_stat'],
                           how='left', coalesce=True))
        fig, axes = plt.subplots(nrows=1, ncols=len(landmarks), figsize=(16, max(8, row_grid.height * 0.4)),
                                 sharex=True, sharey=True, squeeze=False)
        
        axes = axes[0]
        for ax, landmark in zip(axes, landmarks):
            panel_df = full_grid.filter(pl.col('landmark_days') == landmark)
            for significance, color in colors.items():
                points = panel_df.filter((pl.col('significance') == significance)
                                         & pl.col('hazard_ratio_per_sd').is_not_null()
                                         & pl.col('ci_lower').is_not_null()
                                         & pl.col('ci_upper').is_not_null())
                if points.is_empty():
                    continue
                hr = points['hazard_ratio_per_sd'].to_numpy()
                lower = points['ci_lower'].to_numpy()
                upper = points['ci_upper'].to_numpy()
                
                ax.errorbar(x=hr, y=points['y'].to_numpy(), xerr=[hr - lower, upper - hr],
                            fmt='o', color=color, ecolor=color, markersize=5, capsize=2, linestyle='none')
            ax.axvline(1, color='black', linestyle='--', linewidth=1)
            
            for boundary in range(len(feature_stats), row_grid.height, len(feature_stats)):
                ax.axhline(boundary - 0.5, color='gray', linewidth=0.75, alpha=0.5)
            
            ax.set_xlim(0.5, 1.5)
            ax.set_title(f'{landmark}-day landmark')
            ax.set_xlabel('Hazard ratio per SD')
            ax.grid(axis='x', alpha=0.1)
        
        axes[0].set_yticks(row_grid['y'].to_list())
        axes[0].set_yticklabels(row_grid['row_label'].to_list())
        axes[0].invert_yaxis()

        legend_handles = [Line2D([0], [0], marker="o", linestyle="none", color=color, label=label) for label, color in colors.items()]

        fig.legend(handles=legend_handles, title="Significance", loc="lower center", ncol=3)
        fig.suptitle(f"{endpoint_val.upper()} - {lab_name}", fontsize=20, fontweight="bold",)
        fig.tight_layout(rect=[0, 0.08, 1, 0.96])
        fig.savefig(FIG_PATH / f"HR_plots/{lab_name}_HRs_{endpoint_val}.png", dpi=300, bbox_inches="tight")
        plt.close(fig)
        
os.makedirs(FIG_PATH / 'HR_plots', exist_ok=True)
plot_HRs(cox_lab_nominal_hits, 'PSA')
plot_HRs(cox_lab_nominal_hits, 'Testosterone')

In [19]:
sign_alignment = (
    cox_lab_nominal_hits
    .filter(pl.col("coef_feature").is_not_null())
    .with_columns(
        pl.when(pl.col("coef_feature") > 0)
        .then(pl.lit("positive"))
        .when(pl.col("coef_feature") < 0)
        .then(pl.lit("negative"))
        .otherwise(pl.lit("zero"))
        .alias("coef_sign")
    )
    .group_by(["lab_name", "feature_stat"])
    .agg(
        pl.len().alias("n_results"),
        (pl.col("coef_sign") == "positive").sum().alias("n_positive"),
        (pl.col("coef_sign") == "negative").sum().alias("n_negative"),
        (pl.col("coef_sign") == "zero").sum().alias("n_zero"),
        pl.col("coef_sign").n_unique().alias("n_unique_signs"),
        pl.col("cohort").n_unique().alias("n_cohorts"),
        pl.col("landmark_days").n_unique().alias("n_landmarks"),
        pl.col("endpoint").n_unique().alias("n_endpoints"),
    )
    .with_columns(
        (
            (pl.col("n_positive") == pl.col("n_results"))
            | (pl.col("n_negative") == pl.col("n_results"))
        ).alias("sign_aligned"),

        pl.when(pl.col("n_positive") == pl.col("n_results"))
        .then(pl.lit("positive"))
        .when(pl.col("n_negative") == pl.col("n_results"))
        .then(pl.lit("negative"))
        .otherwise(pl.lit("mixed"))
        .alias("aligned_direction"),

        (
            pl.max_horizontal("n_positive", "n_negative")
            / pl.col("n_results")
        ).alias("majority_sign_fraction"),
    )
    .sort(
        ["sign_aligned", "majority_sign_fraction"],
        descending=[True, True],
    )
)

In [20]:
sign_alignment

lab_name,feature_stat,n_results,n_positive,n_negative,n_zero,n_unique_signs,n_cohorts,n_landmarks,n_endpoints,sign_aligned,aligned_direction,majority_sign_fraction
str,str,u32,u32,u32,u32,u32,u32,u32,u32,bool,str,f64
"""Heart rate""","""mean""",10,10,0,0,1,2,3,3,true,"""positive""",1.0
"""Albumin""","""min""",6,0,6,0,1,2,3,1,true,"""negative""",1.0
"""MCHC""","""delta""",3,0,3,0,1,2,2,1,true,"""negative""",1.0
"""Neutrophils absolute""","""min""",2,2,0,0,1,2,1,1,true,"""positive""",1.0
"""Sodium""","""mean""",3,0,3,0,1,1,3,1,true,"""negative""",1.0
…,…,…,…,…,…,…,…,…,…,…,…,…
"""Heart rate""","""delta""",3,2,1,0,2,2,1,2,false,"""mixed""",0.666667
"""WBC""","""delta""",11,4,7,0,2,3,3,3,false,"""mixed""",0.636364
"""PSA""","""delta""",7,3,4,0,2,3,3,2,false,"""mixed""",0.571429


In [21]:
sign_cols = ['lab_name', 'feature_stat', 'n_results', 'n_positive', 'n_negative',
             'sign_aligned', 'aligned_direction', 'majority_sign_fraction']
sign_alignment.filter(pl.col('lab_name') == 'PSA')[sign_cols]

lab_name,feature_stat,n_results,n_positive,n_negative,sign_aligned,aligned_direction,majority_sign_fraction
str,str,u32,u32,u32,bool,str,f64
"""PSA""","""mean""",10,10,0,true,"""positive""",1.0
"""PSA""","""max""",11,11,0,true,"""positive""",1.0
"""PSA""","""last""",10,10,0,true,"""positive""",1.0
"""PSA""","""min""",5,5,0,true,"""positive""",1.0
"""PSA""","""delta""",7,3,4,false,"""mixed""",0.571429


In [22]:
sign_alignment.filter(pl.col('lab_name') == 'Testosterone')[sign_cols]

lab_name,feature_stat,n_results,n_positive,n_negative,sign_aligned,aligned_direction,majority_sign_fraction
str,str,u32,u32,u32,bool,str,f64
"""Testosterone""","""min""",13,0,13,true,"""negative""",1.0
"""Testosterone""","""delta""",3,3,0,true,"""positive""",1.0
"""Testosterone""","""max""",12,0,12,true,"""negative""",1.0
"""Testosterone""","""mean""",15,0,15,true,"""negative""",1.0
"""Testosterone""","""last""",11,0,11,true,"""negative""",1.0


In [23]:
(cox_lab_nominal_hits
 .filter((pl.col('lab_name') == 'PSA') &
         (pl.col('feature_stat') == 'delta'))
 .with_columns(pl.col('p_value').round(3), pl.col('q_value').round(3))
 ['cohort', 'endpoint', 'landmark_days', 'lab_name', 'feature_stat', 'coef_feature', 'p_value', 'q_value'])

cohort,endpoint,landmark_days,lab_name,feature_stat,coef_feature,p_value,q_value
str,str,i16,str,str,f64,f64,f64
"""adt""","""platinum""",90,"""PSA""","""delta""",-0.09061,0.001,0.004
"""adt""","""platinum""",180,"""PSA""","""delta""",-0.094668,0.001,0.006
"""adt""","""avpc""",90,"""PSA""","""delta""",0.16057,0.004,0.035
"""adt_localized""","""avpc""",0,"""PSA""","""delta""",0.17825,0.013,1.0
"""adt_metastatic""","""platinum""",90,"""PSA""","""delta""",-0.093988,0.005,0.022
"""adt_metastatic""","""platinum""",180,"""PSA""","""delta""",-0.10287,0.004,0.02
"""adt_metastatic""","""avpc""",90,"""PSA""","""delta""",0.19735,0.001,0.016


### Somatic Hits

In [24]:
UNI_LAB_SCHEMA = {'cohort' : pl.String,
                  'endpoint' : pl.String,
                  'lab_name' : pl.String,
                  'coef_feature' : pl.Float64,
                  'hazard_ratio_per_sd' : pl.Float64,
                  # 'ci_lower' : pl.Float64,
                  # 'ci_upper' : pl.Float64,
                  'p_value' : pl.Float64,
                  'q_value' : pl.Float64,}

cox_feat_nominal_hits = (pl.scan_csv(DATA_PATH / 'survival_analysis/local_runs_adt*/cox_somatic_gleason/nominally_significant_univariate_results.csv',
                                     glob=True, schema_overrides=UNI_LAB_SCHEMA)
                         .filter(pl.col('endpoint').is_in(['platinum', 'avpc', 'nepc']))
                         .select(UNI_LAB_SCHEMA.keys())
                         .with_columns(pl.col('coef_feature').round(3),
                                       pl.col('hazard_ratio_per_sd').round(3))
                         .rename({'coef_feature' : 'log(HR)', 'hazard_ratio_per_sd' : 'HR'})
                         .collect(engine='streaming'))

nominal_feat_hits_counts = (cox_feat_nominal_hits
                            .group_by(['cohort', 'endpoint'])
                            .len(name='num_nominal_hits')
                            .sort('num_nominal_hits', descending=True))
corrected_feat_hits_counts = (cox_feat_nominal_hits.filter(pl.col('q_value').cast(pl.Float64) < 0.05)
                              .group_by(['cohort', 'endpoint'])
                              .len(name='num_corrected_hits')
                              .sort('num_corrected_hits', descending=True))

full_feat_hits_counts = (nominal_feat_hits_counts
                         .join(corrected_feat_hits_counts, on=['cohort', 'endpoint'], 
                               how='left', coalesce=True)
                         .with_columns(pl.col('num_corrected_hits').fill_null(0))
                         .sort('num_corrected_hits', descending=True))

In [25]:
cox_feat_corrected_hits = cox_feat_nominal_hits.filter(pl.col('q_value') < 0.05)
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=1000, tbl_width_chars=-1,):
    print(cox_feat_corrected_hits.filter(pl.col('lab_name') == 'GLEASON_SCORE'))

shape: (6, 7)
┌────────────────┬──────────┬───────────────┬─────────┬───────┬────────────┬────────────┐
│ cohort         ┆ endpoint ┆ lab_name      ┆ log(HR) ┆ HR    ┆ p_value    ┆ q_value    │
│ ---            ┆ ---      ┆ ---           ┆ ---     ┆ ---   ┆ ---        ┆ ---        │
│ str            ┆ str      ┆ str           ┆ f64     ┆ f64   ┆ f64        ┆ f64        │
╞════════════════╪══════════╪═══════════════╪═════════╪═══════╪════════════╪════════════╡
│ adt            ┆ platinum ┆ GLEASON_SCORE ┆ 0.734   ┆ 2.083 ┆ 7.8874e-13 ┆ 7.8874e-13 │
│ adt            ┆ avpc     ┆ GLEASON_SCORE ┆ 0.411   ┆ 1.509 ┆ 4.9362e-12 ┆ 4.9362e-12 │
│ adt_metastatic ┆ platinum ┆ GLEASON_SCORE ┆ 0.673   ┆ 1.96  ┆ 6.1725e-10 ┆ 6.1725e-10 │
│ adt_metastatic ┆ avpc     ┆ GLEASON_SCORE ┆ 0.435   ┆ 1.545 ┆ 1.1414e-10 ┆ 1.1414e-10 │
│ adt_metastatic ┆ nepc     ┆ GLEASON_SCORE ┆ 0.457   ┆ 1.58  ┆ 0.004647   ┆ 0.004647   │
│ adt            ┆ nepc     ┆ GLEASON_SCORE ┆ 0.307   ┆ 1.359 ┆ 0.024314   ┆ 0.024314 

In [26]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=1000, tbl_width_chars=-1,):
    print(cox_feat_corrected_hits.filter(pl.col('lab_name').str.contains('_SNV')))

shape: (17, 7)
┌────────────────┬──────────┬─────────────┬─────────┬───────┬──────────┬──────────┐
│ cohort         ┆ endpoint ┆ lab_name    ┆ log(HR) ┆ HR    ┆ p_value  ┆ q_value  │
│ ---            ┆ ---      ┆ ---         ┆ ---     ┆ ---   ┆ ---      ┆ ---      │
│ str            ┆ str      ┆ str         ┆ f64     ┆ f64   ┆ f64      ┆ f64      │
╞════════════════╪══════════╪═════════════╪═════════╪═══════╪══════════╪══════════╡
│ adt            ┆ platinum ┆ OGG1_SNV    ┆ 0.183   ┆ 1.201 ┆ 0.000032 ┆ 0.008219 │
│ adt            ┆ platinum ┆ TP53_SNV    ┆ 0.416   ┆ 1.516 ┆ 0.000034 ┆ 0.008219 │
│ adt            ┆ platinum ┆ MYBL1_SNV   ┆ 0.187   ┆ 1.206 ┆ 0.000113 ┆ 0.013948 │
│ adt            ┆ platinum ┆ POLH_SNV    ┆ 0.196   ┆ 1.216 ┆ 0.000115 ┆ 0.013948 │
│ adt            ┆ platinum ┆ SMARCB1_SNV ┆ 0.168   ┆ 1.183 ┆ 0.000148 ┆ 0.014304 │
│ adt            ┆ platinum ┆ RAC1_SNV    ┆ 0.177   ┆ 1.193 ┆ 0.000194 ┆ 0.015624 │
│ adt            ┆ platinum ┆ MRE11A_SNV  ┆ 0.223   ┆ 1.249 ┆

In [27]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=1000, tbl_width_chars=-1,):
    print(cox_feat_nominal_hits.filter((pl.col('lab_name') == 'TP53_SNV') |
                                       (pl.col('lab_name') == 'RB1_SNV') |
                                       (pl.col('lab_name') == 'PTEN_SNV')))

shape: (9, 7)
┌────────────────┬──────────┬──────────┬─────────┬───────┬──────────┬──────────┐
│ cohort         ┆ endpoint ┆ lab_name ┆ log(HR) ┆ HR    ┆ p_value  ┆ q_value  │
│ ---            ┆ ---      ┆ ---      ┆ ---     ┆ ---   ┆ ---      ┆ ---      │
│ str            ┆ str      ┆ str      ┆ f64     ┆ f64   ┆ f64      ┆ f64      │
╞════════════════╪══════════╪══════════╪═════════╪═══════╪══════════╪══════════╡
│ adt            ┆ platinum ┆ TP53_SNV ┆ 0.416   ┆ 1.516 ┆ 0.000034 ┆ 0.008219 │
│ adt            ┆ platinum ┆ PTEN_SNV ┆ 0.235   ┆ 1.265 ┆ 0.002389 ┆ 0.088935 │
│ adt            ┆ avpc     ┆ TP53_SNV ┆ 0.466   ┆ 1.594 ┆ 0.000002 ┆ 0.00095  │
│ adt            ┆ avpc     ┆ RB1_SNV  ┆ 0.166   ┆ 1.181 ┆ 0.043008 ┆ 0.999415 │
│ adt_metastatic ┆ platinum ┆ TP53_SNV ┆ 0.333   ┆ 1.395 ┆ 0.001464 ┆ 0.058946 │
│ adt_metastatic ┆ avpc     ┆ TP53_SNV ┆ 0.397   ┆ 1.488 ┆ 0.000312 ┆ 0.066611 │
│ adt_metastatic ┆ nepc     ┆ RB1_SNV  ┆ 0.381   ┆ 1.464 ┆ 0.001563 ┆ 0.307085 │
│ adt         

### Longitudinal PSA/Testosterone

In [28]:
adt_pred_data = (pl.scan_csv(DATA_PATH / 'longitudinal_prediction_data_adt.csv')
                 .filter(pl.col('LAB_NAME').is_in(['PSA', 'Testosterone']))
                 .select(['DFCI_MRN', 'DIAGNOSIS_DATE', 'LAST_CONTACT_DATE', 'DEATH', 
                          'PLATINUM_DATE', 'PLATINUM', 'TREATMENT_ANCHOR_DATE', 'LAB_DATE',
                          'LAB_NAME', 'LAB_VALUE', 'NEPC', 'NEPC_DATE', 'AVPC', 'AVPC_DATE'])
                 .with_columns(((pl.col('LAB_DATE').str.to_datetime(format='%Y-%m-%d')) - 
                                (pl.col('TREATMENT_ANCHOR_DATE').str.to_datetime(format='%Y-%m-%d')))
                               .dt.total_days().alias('t_lab_rel_adt'))
                 .with_columns(pl.when(pl.col("DFCI_MRN").is_in(metastatic_adt_df["DFCI_MRN"].to_list()))
                               .then(pl.lit("Met"))
                               .otherwise(pl.lit("Local"))
                               .alias("METASTATIC_ADT"))
                 .collect(engine='streaming'))

In [31]:
def generate_lab_trajectory(lab_df, lbound=-2*365, ubound=5*365, bin_width=90, split_features=("PLATINUM", "METASTATIC_ADT"),
                            t_col="t_lab_rel_adt", value_col="LAB_VALUE", log_scale=False):
    
    n_bins = math.ceil((ubound - lbound) / bin_width)
    group_cols = list(split_features)

    filtered_df = (lab_df
                   .filter(pl.col(t_col).is_between(lbound, ubound, closed="both") & 
                           pl.col(value_col).is_not_null() & 
                           pl.col(value_col).is_not_nan() & 
                           pl.all_horizontal([pl.col(col).is_not_null() for col in split_features])))

    if log_scale:
        filtered_df = (filtered_df
                       .filter(pl.col(value_col) > 0)
                       .with_columns(pl.col(value_col).log().alias(value_col)))

    patient_counts = (filtered_df
                      .group_by(group_cols)
                      .agg(pl.col("DFCI_MRN").n_unique().alias("n_patients")))

    window_df = (filtered_df
                 .with_columns(((pl.col(t_col) - lbound) / bin_width).floor()
                               .cast(pl.Int64).clip(0, n_bins - 1).alias("bin"))
                 .group_by(["bin", *group_cols])
                 .agg(pl.len().alias("n_observations"), 
                      pl.col("DFCI_MRN").n_unique().alias("n_patients_bin"), 
                      pl.col(value_col).mean().alias("mean"), 
                      pl.col(value_col).std().alias("std"))
                 .join(patient_counts, on=group_cols, how="left")
                 .with_columns((lbound + pl.col("bin") * bin_width).alias("bin_lower"),
                               (lbound + (pl.col("bin") + 1) * bin_width).clip(upper_bound=ubound).alias("bin_upper"),
                               (pl.col("std") / pl.col("n_observations").sqrt()).alias("se"))
                 .with_columns((pl.col("mean") - 1.96 * pl.col("se")).alias("ci_lower"), 
                               (pl.col("mean") + 1.96 * pl.col("se")).alias("ci_upper"), 
                               ((pl.col("bin_lower") + pl.col("bin_upper")) / 2).alias("bin_midpoint"))
                 .sort(["bin", *group_cols]))

    return window_df

def generate_cohort_trajectories(lab_df, endpoint_col, log_scale=False,
                                 disease_col="METASTATIC_ADT", local_value="Local",
                                 metastatic_value="Met"):
    cohort_dfs = [("Full ADT", lab_df),
                  ("Metastatic ADT", lab_df.filter(pl.col(disease_col) == metastatic_value)),
                  ("Local ADT", lab_df.filter(pl.col(disease_col) == local_value))]

    return pl.concat([generate_lab_trajectory(cohort_df, split_features=(endpoint_col,), log_scale=log_scale)
                      .with_columns(pl.lit(cohort_label).alias("ADT_COHORT"))
                      for cohort_label, cohort_df in cohort_dfs])

def plot_trajectory(trajectory_df, fig_title, png_name, endpoint_col="PLATINUM",
                    cohort_col="ADT_COHORT"):

    if endpoint_col == 'PLATINUM':
        endpoint_label = endpoint_col.capitalize()
    else:
        endpoint_label = endpoint_col
        
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 5), sharex=True, sharey=True)
    cohort_order = ["Full ADT", "Metastatic ADT", "Local ADT"]
    event_specs = [(False, f"Non-{endpoint_label}"), (True, endpoint_label)]
    colors = sns.color_palette(n_colors=2)

    for ax, cohort_label in zip(axes, cohort_order):
        panel_df = trajectory_df.filter(pl.col(cohort_col) == cohort_label)

        for (event_value, group_label), color in zip(event_specs, colors):
            group_df = panel_df.filter(pl.col(endpoint_col).cast(pl.Boolean) == event_value)
            if group_df.is_empty():
                continue

            n_patients = group_df['n_patients'].first()
            
            label = f'{group_label} (N={n_patients:,})'
            
            group_df = group_df.sort("bin_midpoint")
            sns.lineplot(data=group_df, x="bin_midpoint", y="mean", marker="o",
                         color=color, label=label, ax=ax)
            ax.fill_between(group_df["bin_midpoint"].to_numpy(), group_df["ci_lower"].to_numpy(), group_df["ci_upper"].to_numpy(), color=color, alpha=0.2)

        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.set(title=cohort_label, xlabel="Time relative to ADT (days)", ylabel="Mean value")
        ax.legend(title=None)
        sns.despine(ax=ax)

    fig.suptitle(fig_title, fontsize=16, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_PATH / png_name, dpi=300, bbox_inches="tight")
    plt.close(fig)
    
os.makedirs(FIG_PATH / 'lab_trajectories/', exist_ok=True)
os.makedirs(FIG_PATH / 'lab_trajectories_log_scale/', exist_ok=True)
for lab in ['PSA', 'Testosterone']:
    lab_data = adt_pred_data.filter(pl.col('LAB_NAME') == lab)    
    for endpoint in ['PLATINUM', 'NEPC', 'AVPC']:
        traj_df = generate_cohort_trajectories(lab_data, endpoint_col=endpoint)
        log_traj_df = generate_cohort_trajectories(lab_data, endpoint_col=endpoint, log_scale=True)
        plot_trajectory(trajectory_df=traj_df, endpoint_col=endpoint,
                        fig_title=f'{lab} Trajectory for {endpoint}', 
                        png_name=f'lab_trajectories/{lab.lower()}_{endpoint.lower()}_trajectory_three_panel.png')
        plot_trajectory(trajectory_df=log_traj_df, endpoint_col=endpoint,
                        fig_title=f'{lab} Trajectory for {endpoint}', 
                        png_name=f'lab_trajectories_log_scale/{lab.lower()}_{endpoint.lower()}_trajectory_three_panel.png')

### Multivariate Model Metrics

In [85]:
MULTI_METRICS_SCHEMA = {'model' : pl.String, 
                        'cohort' : pl.String, 
                        'endpoint' : pl.String, 
                        'landmark_days' : pl.Int16, 
                        'n_train_val' : pl.Int64, 
                        'n_test' : pl.Int64, 
                        'n_events_train_val' : pl.Int64, 
                        'n_events_test' : pl.Int64, 
                        'test_c_index' : pl.Float64, 
                        'test_mean_auc_t' : pl.Float64}

multi_files = list((DATA_PATH / "survival_analysis").glob("local_runs_adt*/cox/landmark_*/both/cox_agg_multivariable_metrics.csv")) + \
              list((DATA_PATH / 'survival_analysis').glob('local_runs_adt*/xgboost/landmark_*/both/landmark_xgboost_metrics_landmark*.csv'))
multi_metrics = (pl.concat([pl.scan_csv(file) for file in multi_files], how="diagonal_relaxed")
                 .select(MULTI_METRICS_SCHEMA.keys())
                 .cast(MULTI_METRICS_SCHEMA)
                 .collect(engine="streaming"))

endpoints = ['platinum', 'nepc', 'avpc']
cohorts = ['all', 'localized', 'metastatic']
landmarks = [0, 90, 180]
models = ['elastic_net_cox', 'xgboost']

os.makedirs(FIG_PATH / f'multi_metrics/', exist_ok=True)
for endpoint in endpoints:
    endpoint_df = multi_metrics.filter(pl.col('endpoint') == endpoint)

    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(14,6), sharey=True)
    handles, labels = None, None
    for ax, cohort in zip(axes, cohorts): 
        panel_df = endpoint_df.filter(pl.col('cohort') == cohort)

        n_train_val, n_test, n_events_train_val, n_events_test = (panel_df.filter(pl.col('landmark_days') == 0)
                                                                 ['n_train_val', 'n_test', 'n_events_train_val', 'n_events_test'][0].to_numpy()[0])

        sns.barplot(data=panel_df, x='landmark_days', y='test_c_index', hue='model',
                    order=landmarks, hue_order=models, errorbar=None, ax=ax)
        ax.axhline(y=0.5, color='red', linestyle=':', linewidth=1.5, zorder=1)
        ax.set(title=f'{cohort.capitalize()} (N Px\'s={n_train_val + n_test}, N Events = {n_events_train_val + n_events_test})', 
               xlabel='Landmark (days)', ylabel='Test C-Index' if ax is axes[0] else '')

        if handles is None:
            handles, labels = ax.get_legend_handles_labels()
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.legend(handles, labels, title=None, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=len(models), frameon=False)
    fig.suptitle(endpoint.upper(), fontsize=14, fontweight='bold')
    fig.tight_layout()
    fig.savefig(FIG_PATH / f'multi_metrics/{endpoint.lower()}_metrics_by_cohort.png', dpi=300, bbox_inches="tight")
    plt.close(fig)